# Entregable 1 · Ayuda automática desde Stack Overflow

Primero aparece el enunciado con sus ejemplos y después la solución: la función `get_SOS_help` y sus pruebas.
Ejecuta las celdas en orden (*Run All*). Se necesita conexión a Internet.

## Web scraping exercise

Define a generic function `SOS_help` which retrieves help results from Stack Overflow Stunning results. <br>

The following command works just fine:

In [1]:
import numpy as np
np.random.uniform(-1, 1, size=100)

array([ 0.17277112, -0.45982208, -0.77532774,  0.80524879, -0.14479289,
        0.26082433, -0.23281354,  0.0651473 ,  0.0934179 , -0.71085971,
       -0.31573988, -0.62679726,  0.60225502,  0.00654189,  0.40267064,
        0.13619718, -0.616806  ,  0.65519418, -0.50125576,  0.07451455,
        0.29363925, -0.33829479,  0.33674105, -0.97032261,  0.82155008,
       -0.81667484,  0.71739785,  0.68141862, -0.65349203,  0.81150411,
        0.82879261, -0.95452256,  0.22343723, -0.12908977,  0.17524704,
       -0.68875759,  0.34956507, -0.47197434,  0.6224921 ,  0.01317966,
        0.28231355,  0.61420974,  0.73146461,  0.92869374,  0.41312283,
        0.25469193, -0.56672422, -0.43688542,  0.76513551,  0.90377152,
        0.50479289,  0.0022284 , -0.50302897,  0.8152439 ,  0.51828048,
       -0.60131848, -0.73023345,  0.67894224,  0.35608675, -0.88981347,
       -0.84845279, -0.09517343,  0.1368883 ,  0.84605281,  0.96497816,
       -0.21579629,  0.61241903, -0.74915295,  0.0180831 , -0.39

`print_out` prints command execution

In [2]:
def print_output(command):
    command = 'print(' + command + ')'
    return exec(command)

In [3]:
print_output(command = 'np.random.uniform(-1, 1, size=100)')

[-0.89140376 -0.97817255  0.31132112 -0.50769688  0.7668566   0.97969087
  0.04367668 -0.2332837  -0.70800225 -0.40808094  0.19752683 -0.86566936
 -0.61506424 -0.8464156  -0.15834153 -0.22041297 -0.87586981  0.05470385
 -0.81463104  0.34153264 -0.48319779 -0.10432507  0.0489679   0.19772266
  0.86223104  0.87109101  0.58360809  0.52060304 -0.36723781 -0.0469913
  0.33081883 -0.67327643 -0.00808793  0.81621463 -0.89220335  0.93383158
  0.63241038 -0.11210031  0.31294415  0.32633301  0.26856545 -0.39726355
 -0.05457143  0.12238069 -0.41985845  0.91535091  0.04608178 -0.70334479
  0.10015875 -0.02485197 -0.75236366  0.52263586  0.08569489 -0.61670275
 -0.39968448 -0.0255691  -0.57079166  0.61889287 -0.78561029  0.4733222
 -0.92584152  0.86063642 -0.13361194 -0.1380942  -0.63709812 -0.88278407
  0.18016215  0.2939474   0.62004877  0.03098843  0.28166548  0.58526144
  0.01830484 -0.36774545  0.46428082 -0.96557822 -0.19221324  0.39838614
  0.30395766 -0.74172145 -0.8921677  -0.91524362 -0.8

if `print_out` results in an error, this is printed

In [4]:
# Error intencionado: se captura para que el cuaderno siga ejecutándose.
try:
    print_output(command = 'nn.random.uniform(-1, 1, size=100)')
except Exception as error:
    print(f"{type(error).__name__}: {error}")

NameError: name 'nn' is not defined


In [5]:
# Error intencionado: se captura para que el cuaderno siga ejecutándose.
try:
    print_output(command = 'np.random.uniform(-1, 1, siz=100)')
except Exception as error:
    print(f"{type(error).__name__}: {error}")

TypeError: uniform() got an unexpected keyword argument 'siz'


Create a function `get_SOS_help` which: <br>
    - Prints "works as intended" if no error. <br>
    - Prints the first link from stack overflow related to the error. As an example: <br>
        `print_output(command = 'np.random.uniform(-1, 1, siz=100)'`
        should retrieve the following link:
        https://stackoverflow.com/questions/72537485/typeerror-uniform-got-an-unexpected-keyword-argument-low-size <br>
    - Prints the most voted help
    - Opens a new browser using the link

## Solución

`get_SOS_help(command)` sigue los pasos del enunciado:

1. **Ejecuta** `command` con `exec`. Si no hay error, imprime `works as intended`.
2. **Busca el error** en Stack Overflow y muestra el **primer enlace**.
3. Muestra la **respuesta más votada** de esa pregunta.
4. **Abre el enlace** en una nueva ventana del navegador.

**Cómo se obtienen los datos.** Stack Overflow no deja descargar automáticamente sus páginas de búsqueda
(pide un captcha) ni las de preguntas (responde con error 403). Por eso los datos se piden a la
[API oficial de Stack Exchange](https://api.stackexchange.com/docs), que devuelve los mismos contenidos
en formato JSON. La respuesta llega en HTML y **BeautifulSoup** la convierte en texto legible.

Antes de buscar se eliminan del mensaje los valores entre comillas, como `'siz'`. Así la búsqueda trata
el error en general (`uniform() got an unexpected keyword argument`) y encuentra el enlace del enunciado.

In [6]:
# Si falta alguna librería: %pip install numpy requests beautifulsoup4
import re
import webbrowser

import numpy as np
import requests
from bs4 import BeautifulSoup

In [7]:
API = "https://api.stackexchange.com/2.3"


def get_SOS_help(command):
    # 1. Ejecutar el código. Si funciona, no hace falta buscar ayuda.
    try:
        exec(command)
        print("works as intended")
        return
    except Exception as error:
        print(f"{type(error).__name__}: {error}")
        query = re.sub(r"'.*?'", "", str(error))  # quita valores concretos, como 'siz'

    # 2. Primer enlace de Stack Overflow relacionado con el error.
    questions = requests.get(f"{API}/search/advanced", params={
        "q": query, "tagged": "python", "site": "stackoverflow",
        "sort": "relevance", "pagesize": 1,
    }).json()["items"]
    if not questions:
        print("No se ha encontrado ninguna pregunta relacionada.")
        return
    question = questions[0]
    print("\nPrimer enlace:", question["link"])

    # 3. Respuesta más votada de esa pregunta.
    answers = requests.get(f"{API}/questions/{question['question_id']}/answers", params={
        "site": "stackoverflow", "sort": "votes", "pagesize": 1, "filter": "withbody",
    }).json()["items"]
    print("\nRespuesta más votada:")
    if answers:
        print(BeautifulSoup(answers[0]["body"], "html.parser").get_text().strip())
    else:
        print("La pregunta todavía no tiene respuestas.")

    # 4. Abrir la pregunta en una nueva ventana del navegador.
    webbrowser.open_new(question["link"])

### Prueba 1 · Código correcto

Debe imprimir `works as intended`, sin buscar en Internet ni abrir el navegador.

In [8]:
command = 'np.random.uniform(-1, 1, size=100)'
get_SOS_help(command)

works as intended


### Prueba 2 · El ejemplo del enunciado

`siz` en lugar de `size`. Debe mostrar el enlace
https://stackoverflow.com/questions/72537485/typeerror-uniform-got-an-unexpected-keyword-argument-low-size,
su respuesta más votada y abrirlo en el navegador.

In [9]:
command = 'np.random.uniform(-1, 1, siz=100)'
get_SOS_help(command)

TypeError: uniform() got an unexpected keyword argument 'siz'

Primer enlace: https://stackoverflow.com/questions/72537485/typeerror-uniform-got-an-unexpected-keyword-argument-low-size

Respuesta más votada:
We must use random.uniform while importing the related class with an alias (using 'as') or else just use import numpy while importing
An example for using alias is :
from numpy import random as np_random

Then utilize np_random.uniform()
(figured it out and had to use this to solve the problem - note I ran it on colab)


## Create a malfunctioning code and use this function on it

### Prueba 3 · Código propio que falla

Convertir a número un texto que no lo es (`int("abc")`) produce un `ValueError`.

In [10]:
command = 'int("abc")'
get_SOS_help(command)

ValueError: invalid literal for int() with base 10: 'abc'

Primer enlace: https://stackoverflow.com/questions/1841565/valueerror-invalid-literal-for-int-with-base-10

Respuesta más votada:
The error message means that the string provided to int could not be parsed as an integer. The part at the end, after the :, shows the string that was provided.
In the case described in the question, the input was an empty string, written as ''.
Here is another example - a string that represents a floating-point value cannot be converted directly with int:
>>> int('55063.000000')
Traceback (most recent call last):
  File "<stdin>", line 1, in <module>
ValueError: invalid literal for int() with base 10: '55063.000000'

Instead, convert to float first:
>>> int(float('55063.000000'))
55063
